# 04 - Preparacion de la matriz de modelado

Prepara una matriz numerica limpia, imputada, transformada y escalada a partir de
`data/features/dataset_personas_features.csv` (1 fila = 1 persona, generado en
`03_construccion_features.ipynb`).

**Alcance:** unicamente preparacion de datos para clustering

**Principio:** cada decision de imputacion, transformacion o codificacion debe quedar
documentada y ser explicable en la tesis; ninguna se aplica de forma automatica o
silenciosa cuando depende de la semantica de los datos.

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, RobustScaler, StandardScaler
import joblib

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent.parent
FEATURES_DIR = PROJECT_ROOT / "data" / "features"
MODELING_DIR = PROJECT_ROOT / "data" / "modeling"
MODELING_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 60)
pd.set_option("display.max_rows", 120)
pd.set_option("display.width", 200)

## 1. Inspeccion inicial

In [ ]:
dataset_personas_features = pd.read_csv(FEATURES_DIR / "dataset_personas_features.csv")
df = dataset_personas_features

print("Filas (personas):", df.shape[0])
print("Columnas:", df.shape[1])
print("IDPERSONA unicos:", df["IDPERSONA"].nunique())
df.dtypes.value_counts()

In [ ]:
def resumen_columnas(data: pd.DataFrame) -> pd.DataFrame:
    """Tipo, % nulos, valores unicos y participacion del valor dominante por columna."""
    filas = []
    for c in data.columns:
        vc = data[c].value_counts(dropna=True, normalize=True)
        filas.append({
            "FEATURE": c, "DTYPE": str(data[c].dtype),
            "PCT_NULOS": round(data[c].isna().mean() * 100, 2),
            "N_UNICOS": data[c].nunique(dropna=True),
            "PCT_VALOR_DOMINANTE": round(vc.iloc[0] * 100, 2) if len(vc) else np.nan,
        })
    out = pd.DataFrame(filas).set_index("FEATURE")
    out["CONSTANTE"] = out["N_UNICOS"] <= 1
    out["CASI_CONSTANTE"] = out["PCT_VALOR_DOMINANTE"] >= 97
    return out

resumen_inicial = resumen_columnas(df)
resumen_inicial

In [ ]:
print("Columnas constantes:", resumen_inicial["CONSTANTE"].sum())
print("Columnas casi constantes (>=97% valor dominante):", resumen_inicial["CASI_CONSTANTE"].sum())
resumen_inicial[resumen_inicial["CASI_CONSTANTE"]]

In [ ]:
num_dtype_cols = df.select_dtypes(include=["int64", "float64"]).columns.tolist()
n_inf = np.isinf(df[num_dtype_cols].to_numpy()).sum()
print("Valores infinitos en columnas numericas:", n_inf)
assert n_inf == 0

### Verificacion de IDPERSONA

`IDPERSONA` debe ser unico y se conserva unicamente como identificador; no participa
como feature del clustering.

In [ ]:
assert df["IDPERSONA"].is_unique, "IDPERSONA no es unico"
assert df["IDPERSONA"].isna().sum() == 0
print("IDPERSONA es unico y no tiene nulos. OK.")

## 2. Clasificacion de variables

Clasificacion manual y documentada (no se infiere el tipo automaticamente de forma
ciega): identificador, numericas, binarias, categoricas nominales, categorica ordinal
y categoricas de alta cardinalidad (ver seccion 3). No quedo ninguna columna de texto
libre, URL, nombre de archivo o metadata administrativa en este dataset: esas columnas
ya fueron excluidas en `03_construccion_features.ipynb` (ver
`data/features/features_excluded.csv`).

In [ ]:
ID_COL = "IDPERSONA"

BINARY_COLS = [
    "MULTIPLES_CARGOS_MISMO_ANIO", "PASO_POR_RECTORADO",
    "ES_DOCENTE_ADMIN_MIXTO", "VIGENTE_ACTUALMENTE",
]  # True/False (con NaN ocasional) -> se mapean a 0/1

HIGH_CARD_COLS = ["CARGO_ACTUAL", "CARGO_MAS_FRECUENTE", "UNIDAD_ACTUAL_NOMBRE"]  # ver seccion 3

ORDINAL_COLS = ["NIVEL_ACADEMICO_MAXIMO"]  # orden semantico documentado en feature_dictionary.csv

NOMINAL_COLS = [
    "REGIMEN_INICIAL_DESC", "REGIMEN_ACTUAL_DESC", "TIPOEMPLEADO_ACTUAL_DESC",
    "DEDICACION_DOCENTE_ACTUAL", "DEDICACION_DOCENTE_MAS_FRECUENTE",
]  # sin orden claro (p.ej. Medio Tiempo vs Tiempo Parcial no tiene jerarquia inequivoca
   # y existe el valor 'No Aplica'); se tratan como nominales -> One-Hot Encoding

texto_libre_cols = (
    df.select_dtypes(include=["object", "string", "category"])
    .columns.difference([ID_COL] + BINARY_COLS + HIGH_CARD_COLS + ORDINAL_COLS + NOMINAL_COLS)
    .tolist()
)

NUMERIC_COLS = [c for c in df.columns if c != ID_COL
                and c not in BINARY_COLS + HIGH_CARD_COLS + ORDINAL_COLS + NOMINAL_COLS]

assert texto_libre_cols == []
assert set(NUMERIC_COLS) | set(BINARY_COLS) | set(HIGH_CARD_COLS) | set(ORDINAL_COLS) | set(NOMINAL_COLS) \
    == set(df.columns) - {ID_COL}

print(f"ID: 1 | Numericas: {len(NUMERIC_COLS)} | Binarias: {len(BINARY_COLS)} | "
      f"Ordinal: {len(ORDINAL_COLS)} | Nominales: {len(NOMINAL_COLS)} | Alta cardinalidad: {len(HIGH_CARD_COLS)}")

## 3. Variables excluidas

`CARGO_ACTUAL`, `CARGO_MAS_FRECUENTE` y `UNIDAD_ACTUAL_NOMBRE` no son texto libre ni
metadata: son categoricas institucionales legitimas, pero con demasiadas categorias
para One-Hot Encoding (ver cardinalidad abajo). No se inventan agrupaciones; se
excluyen de `X_modelado` y quedan disponibles en el dataset original para un
tratamiento especifico posterior (p.ej. frequency/target encoding revisado por un
experto, o agrupacion institucional validada).

In [ ]:
MAX_CATEGORIAS_ONEHOT = 20  # umbral documentado: evita cientos de columnas dummy dispersas

cardinalidad_alta = pd.DataFrame([
    {"FEATURE": c, "NUM_CATEGORIAS": df[c].nunique(dropna=True),
     "PORCENTAJE_DOMINANTE": round(df[c].value_counts(normalize=True, dropna=True).iloc[0] * 100, 2),
     "DECISION": "EXCLUIR_DE_MATRIZ_TRADICIONAL_REVISAR_LUEGO"}
    for c in HIGH_CARD_COLS
])
assert (cardinalidad_alta["NUM_CATEGORIAS"] > MAX_CATEGORIAS_ONEHOT).all()
cardinalidad_alta

In [ ]:
modelado_variables_excluidas = pd.DataFrame([
    {"FEATURE": c,
     "RAZON": f"Cardinalidad excesivamente alta para One-Hot Encoding ({df[c].nunique()} categorias, "
              f"> umbral de {MAX_CATEGORIAS_ONEHOT}); se conserva en el dataset original para un "
              "tratamiento especifico posterior (frequency/target encoding o agrupacion institucional "
              "revisada por un experto), no se inventa una agrupacion aqui."}
    for c in HIGH_CARD_COLS
])
modelado_variables_excluidas.to_csv(FEATURES_DIR / "modelado_variables_excluidas.csv", index=False)
modelado_variables_excluidas

## 4. Valores nulos

Para cada variable se distingue si el nulo representa informacion desconocida o si,
dado el contexto de otra columna, corresponde a una ausencia real de actividad (que
ya se documenta explicitamente, no se asume). La estrategia se decide por bloques
segun esa evidencia, nunca "NaN -> 0" de forma generalizada.

In [ ]:
null_pct = df.isna().mean().mul(100).round(2).sort_values(ascending=False)
null_pct[null_pct > 0]

In [ ]:
# Evidencia: estas tasas/promedios son NaN exactamente cuando el conteo asociado es 0
# (division indefinida), por lo que 0 es consistente con el resto de la fila.
ZERO_IMPUTE = {
    "PROMEDIO_ESTUDIANTES_POR_CURSO": "NaN ssi NUM_CURSOS=0 (sin docencia, promedio indefinido).",
    "HORAS_PROMEDIO_POR_ACTIVIDAD_POLITECNICA": "NaN ssi NUM_ACTIVIDADES_POLITECNICAS=0.",
    "COBERTURA_EVALUACION": "NaN ssi NUM_EVALUACIONES=0 (sin estudiantes evaluados).",
    "PROPORCION_CAPACITACIONES_APROBADAS": "NaN ssi NUM_CAPACITACIONES=0.",
    "NUM_PUBLICACIONES_POR_ANIO": "NaN ssi ANTIGUEDAD_EFECTIVA_ANIOS<1 (tasa anual no definible); "
        "93% de estos casos ya tiene NUM_PUBLICACIONES=0, el resto conserva su conteo bruto en esa columna.",
    "NUM_PROYECTOS_INVESTIGACION_POR_ANIO": "NaN ssi ANTIGUEDAD_EFECTIVA_ANIOS<1; mayoria con "
        "NUM_PROYECTOS_INVESTIGACION=0, conteo bruto se conserva en esa columna.",
}
for c, cond in [("PROMEDIO_ESTUDIANTES_POR_CURSO", "NUM_CURSOS"),
                ("COBERTURA_EVALUACION", "NUM_EVALUACIONES"),
                ("HORAS_PROMEDIO_POR_ACTIVIDAD_POLITECNICA", "NUM_ACTIVIDADES_POLITECNICAS"),
                ("PROPORCION_CAPACITACIONES_APROBADAS", "NUM_CAPACITACIONES")]:
    mask = df[c].isna()
    assert (df.loc[mask, cond] == 0).all(), f"{c}: NaN no coincide con {cond}==0"

# Variables donde 0 SI sesgaria el significado (score o "ausencia real" no aplica): mediana.
MEDIAN_IMPUTE = {
    "PROMEDIO_HETEROEVALUACION": "Score 44-100; NaN ssi NUM_EVALUACIONES=0. Imputar 0 lo leeria como "
        "'peor evaluacion posible', que es incorrecto: la ausencia de evaluacion ya queda marcada por "
        "NUM_EVALUACIONES=0 y COBERTURA_EVALUACION=0 en otras columnas.",
}
TRAYECTORIA_SIN_HISTORIAL = [
    "N_REGISTROS_HISTORIAL", "N_CONTRATOS_TOTAL", "N_CARGOS_DISTINTOS", "N_UNIDADES_DISTINTAS",
    "N_FACULTADES_DISTINTAS", "N_REGIMENES_DISTINTOS", "N_REINGRESOS", "N_DEDICACIONES_DOCENTE_DISTINTAS",
    "ANIOS_EXPERIENCIA_DOCENTE", "ANIOS_EXPERIENCIA_ADMINISTRATIVO",
    "ANTIGUEDAD_EFECTIVA_ANIOS", "ANTIGUEDAD_CALENDARIO_ANIOS", "PROPORCION_CONTRATOS_FINALIZADOS",
]
for c in TRAYECTORIA_SIN_HISTORIAL:
    MEDIAN_IMPUTE[c] = ("Nulo en las mismas 5 personas (0.23%) sin registros en historial_laboral: "
        "estan en el padron institucional, por lo que la ausencia de registro es una brecha de "
        "integracion (desconocido), no evidencia de 0 actividad; se usa la mediana en vez de 0.")

remaining_numeric = [c for c in NUMERIC_COLS if c not in ZERO_IMPUTE and c not in MEDIAN_IMPUTE]
assert df[remaining_numeric].isna().sum().sum() == 0, "hay numericas sin estrategia de imputacion"
assert set(ZERO_IMPUTE) | set(MEDIAN_IMPUTE) | set(remaining_numeric) == set(NUMERIC_COLS)
print(f"Numericas: {len(ZERO_IMPUTE)} imputadas con 0, {len(MEDIAN_IMPUTE)} con mediana, "
      f"{len(remaining_numeric)} sin nulos.")

In [ ]:
# Categoricas: DESCONOCIDO como categoria explicita (distinta de 'No Aplica', que ya existe
# como valor propio en DEDICACION_DOCENTE_* para casos donde la dedicacion no rige).
CATEGORICA_DESCONOCIDO = {
    "REGIMEN_INICIAL_DESC": "Regimen inicial no siempre registrado en historial_laboral antiguo.",
    "REGIMEN_ACTUAL_DESC": "Alineado con las 5 personas sin historial_laboral.",
    "TIPOEMPLEADO_ACTUAL_DESC": "Alineado con las 5 personas sin historial_laboral.",
    "DEDICACION_DOCENTE_ACTUAL": "Sin registro de docencia para determinar dedicacion; distinto de "
        "'No Aplica', que es un valor explicito ya presente en los datos.",
    "DEDICACION_DOCENTE_MAS_FRECUENTE": "Sin registro de docencia; distinto de 'No Aplica' explicito.",
    "CARGO_ACTUAL": "Se documenta igualmente aunque la variable se excluye de X_modelado (seccion 3).",
    "CARGO_MAS_FRECUENTE": "Se documenta igualmente aunque la variable se excluye de X_modelado.",
    "UNIDAD_ACTUAL_NOMBRE": "Se documenta igualmente aunque la variable se excluye de X_modelado; "
        "ya existe ademas la categoria explicita 'DESCONOCIDA' en los datos originales.",
}
assert set(CATEGORICA_DESCONOCIDO) == set(NOMINAL_COLS) | set(HIGH_CARD_COLS)
assert df["NIVEL_ACADEMICO_MAXIMO"].isna().sum() == 0  # ordinal sin nulos, no requiere estrategia

# Binarias: solo 0.23% de nulos (las mismas 5 personas); se imputan con la moda.
BINARY_MODE_IMPUTE = {c: "5 personas (0.23%) sin historial_laboral; volumen minimo, se usa la moda."
                       for c in BINARY_COLS}
assert set(BINARY_MODE_IMPUTE) == set(BINARY_COLS)
print("Estrategias de imputacion definidas para todas las variables no numericas.")

In [ ]:
def fila_imputacion(c):
    pct = round(df[c].isna().mean() * 100, 2)
    if c in ZERO_IMPUTE:
        return pct, "CONSTANTE", "0", ZERO_IMPUTE[c]
    if c in MEDIAN_IMPUTE:
        return pct, "MEDIANA", str(round(df[c].median(), 3)), MEDIAN_IMPUTE[c]
    if c in CATEGORICA_DESCONOCIDO:
        return pct, "CATEGORIA_DESCONOCIDO", "DESCONOCIDO", CATEGORICA_DESCONOCIDO[c]
    if c in BINARY_MODE_IMPUTE:
        return pct, "MODA", str(df[c].mode(dropna=True).iloc[0]), BINARY_MODE_IMPUTE[c]
    return pct, "SIN_NULOS", "N/A", "Variable sin valores nulos."

imputacion_diagnostico = pd.DataFrame(
    [{"FEATURE": c, "PORCENTAJE_NULOS": p, "ESTRATEGIA": e, "VALOR_IMPUTACION": v, "JUSTIFICACION": j}
     for c in df.columns if c != ID_COL
     for p, e, v, j in [fila_imputacion(c)]]
)
assert len(imputacion_diagnostico) == df.shape[1] - 1
imputacion_diagnostico.to_csv(FEATURES_DIR / "imputacion_diagnostico.csv", index=False)
imputacion_diagnostico.sort_values("PORCENTAJE_NULOS", ascending=False).head(20)

In [ ]:
# Aplicar imputacion sobre una copia; el dataset original NO se modifica.
df_prep = df.copy()
for c in ZERO_IMPUTE:
    df_prep[c] = df_prep[c].fillna(0)
for c in MEDIAN_IMPUTE:
    df_prep[c] = df_prep[c].fillna(df_prep[c].median())
assert df_prep[NUMERIC_COLS].isna().sum().sum() == 0
print("Numericas imputadas. Nulos restantes en numericas:", df_prep[NUMERIC_COLS].isna().sum().sum())

## 5. Variables muy sesgadas

`log1p` se evalua solo sobre variables de conteo/duracion no negativas con
`|skew| >= 2` (mismo umbral usado como referencia en `03_construccion_features.ipynb`).
Se excluyen explicitamente las tasas/promedios/proporciones ya acotadas (0-1 o en una
escala de score), donde `log1p` no tiene una interpretacion clara.

In [ ]:
RATE_LIKE = {  # ya normalizadas / acotadas: log1p no aporta o distorsiona la escala
    "PROMEDIO_ESTUDIANTES_POR_CURSO", "PROMEDIO_HETEROEVALUACION", "COBERTURA_EVALUACION",
    "PROPORCION_CONTRATOS_FINALIZADOS", "PROPORCION_CAPACITACIONES_APROBADAS",
}
SKEW_THRESHOLD = 2.0

skew_before = df_prep[NUMERIC_COLS].skew()
LOG1P_COLS = sorted([c for c in NUMERIC_COLS if c not in RATE_LIKE and skew_before[c] >= SKEW_THRESHOLD])

for c in LOG1P_COLS:
    assert (df_prep[c] >= 0).all(), f"{c} tiene valores negativos, no apto para log1p"
    df_prep[c] = np.log1p(df_prep[c])
skew_after = df_prep[NUMERIC_COLS].skew()

sesgo_diagnostico = pd.DataFrame({
    "FEATURE": NUMERIC_COLS,
    "SKEW_ANTES": skew_before.reindex(NUMERIC_COLS).round(3).values,
    "TRANSFORMACION": ["log1p" if c in LOG1P_COLS else "ninguna" for c in NUMERIC_COLS],
    "SKEW_DESPUES": [round(skew_after[c], 3) if c in LOG1P_COLS else round(skew_before[c], 3) for c in NUMERIC_COLS],
}).sort_values("SKEW_ANTES", key=abs, ascending=False)

print(f"{len(LOG1P_COLS)} de {len(NUMERIC_COLS)} numericas transformadas con log1p "
      f"(|skew| >= {SKEW_THRESHOLD} y no son tasas/proporciones).")
sesgo_diagnostico.head(20)

## 6. Variables categoricas

### 6.1 Nominales vs. ordinal

`NIVEL_ACADEMICO_MAXIMO` tiene un orden semantico documentado en
`feature_dictionary.csv` (derivado de los conteos de titulaciones) y se codifica como
ordinal explicito. Las demas categoricas de baja cardinalidad no tienen un orden
inequivoco (p.ej. `DEDICACION_DOCENTE_*`: 'Medio Tiempo' vs 'Tiempo Parcial' no tiene
jerarquia clara y coexiste con 'No Aplica') y se tratan como nominales con One-Hot
Encoding.

In [ ]:
for c in NOMINAL_COLS + ORDINAL_COLS:
    print(f"--- {c} ({df_prep[c].nunique(dropna=True)} categorias) ---")
    print((df_prep[c].value_counts(dropna=False, normalize=True) * 100).round(2))
    print()

In [ ]:
# Imputacion categorica (definida en seccion 4)
for c in NOMINAL_COLS:
    df_prep[c] = df_prep[c].astype("object").where(df_prep[c].notna(), "DESCONOCIDO").astype(str)
assert df_prep[NOMINAL_COLS].isna().sum().sum() == 0

# Ordinal explicito: jerarquia documentada en feature_dictionary.csv (NIVEL_ACADEMICO_MAXIMO)
NIVEL_ORDEN = {"sin_registro": 0, "primaria": 1, "bachillerato": 2, "tercer_nivel": 3, "cuarto_nivel": 4}
assert set(df_prep["NIVEL_ACADEMICO_MAXIMO"].dropna().unique()) <= set(NIVEL_ORDEN)
df_prep["NIVEL_ACADEMICO_MAXIMO_ORD"] = df_prep["NIVEL_ACADEMICO_MAXIMO"].map(NIVEL_ORDEN)
assert df_prep["NIVEL_ACADEMICO_MAXIMO_ORD"].isna().sum() == 0
print("Categoricas nominales imputadas y ordinal codificada.")

## 7. Variables binarias

Se verifica que solo tomen los valores esperados antes de mapearlas a `0/1`; no se
aplica One-Hot Encoding a variables que ya son binarias.

In [ ]:
for c in BINARY_COLS:
    valores = set(df[c].dropna().unique().tolist())
    assert valores <= {True, False}, f"{c} tiene valores inconsistentes: {valores}"

for c in BINARY_COLS:
    moda = df_prep[c].mode(dropna=True).iloc[0]
    df_prep[c] = df_prep[c].fillna(moda).astype(int)

assert df_prep[BINARY_COLS].isin([0, 1]).all().all()
df_prep[BINARY_COLS].mean().rename("proporcion_en_1").to_frame()

## 9. Variables con muchos ceros

Un porcentaje alto de ceros no se penaliza por si solo: puede ser justamente el rasgo
que distingue un perfil (p.ej. quien no dirige proyectos de vinculacion como director
de programa). Se documenta como diagnostico; no se elimina ninguna variable solo por
esta razon.

In [ ]:
ceros_diagnostico = pd.DataFrame({
    "FEATURE": NUMERIC_COLS,
    "PCT_CEROS": [round((df[c] == 0).mean() * 100, 2) for c in NUMERIC_COLS],
    "VARIANZA": [round(df[c].var(), 3) for c in NUMERIC_COLS],
}).sort_values("PCT_CEROS", ascending=False)

print("Variables con >=90% de ceros (revisadas, se conservan por utilidad conceptual):")
ceros_diagnostico[ceros_diagnostico["PCT_CEROS"] >= 90]

In [ ]:
# Ninguna variable con alto % de ceros tiene varianza 0 (no son constantes): se conservan todas.
casi_constantes_por_ceros = ceros_diagnostico[(ceros_diagnostico["PCT_CEROS"] >= 90) & (ceros_diagnostico["VARIANZA"] == 0)]
assert casi_constantes_por_ceros.empty
print("Ninguna variable de alto % de ceros es constante (varianza=0); todas se conservan.")

## 10. Redundancia (correlaciones)

Pares de variables numericas con `|correlacion| >= 0.85` (calculada sobre el bloque ya
imputado y transformado con `log1p`). No se elimina ninguna automaticamente: se
distingue entre pares parte-todo / numerador-denominador (conceptualmente distintos,
aportan granularidad) y posibles duplicados practicos, para que la decision final de
eliminar quede como una revision explicita de tesis.

In [ ]:
CORR_THRESHOLD = 0.85
corr = df_prep[NUMERIC_COLS].corr()
pares = corr.where(np.triu(np.ones(corr.shape), 1).astype(bool)).stack()
pares = pares[pares.abs() >= CORR_THRESHOLD].sort_values(ascending=False)

correlaciones_features = pares.reset_index()
correlaciones_features.columns = ["VARIABLE_1", "VARIABLE_2", "CORRELACION"]
correlaciones_features["CORRELACION"] = correlaciones_features["CORRELACION"].round(3)
correlaciones_features["POSIBLE_DUPLICADO_PRACTICO"] = correlaciones_features["CORRELACION"] >= 0.99

correlaciones_features.to_csv(FEATURES_DIR / "correlaciones_features.csv", index=False)
print(f"{len(correlaciones_features)} pares con |corr| >= {CORR_THRESHOLD}; "
      f"{correlaciones_features['POSIBLE_DUPLICADO_PRACTICO'].sum()} con corr >= 0.99 (revisar con prioridad).")
correlaciones_features.head(15)

## 11. Multicolinealidad

Ningun par alcanza una correlacion de duplicado exacto (`>=0.99`) en este dataset
(los duplicados deterministas ya fueron eliminados en `03_construccion_features.ipynb`,
ver `features_excluded.csv`). Los pares mas altos representan la misma dimension
mirada desde dos angulos utiles y **se conservan ambos** en `X_modelado`, dejando la
decision de fusionarlos para una revision de tesis explicita:

- `ANTIGUEDAD_EFECTIVA_ANIOS` vs `ANTIGUEDAD_CALENDARIO_ANIOS`: tiempo activo vs. tiempo
  calendario (incluye interrupciones); la diferencia entre ambas es la señal de reingresos.
- `TOTAL_REGISTRADOS` vs `TOTAL_EVALUADOS`: numerador y denominador de
  `COBERTURA_EVALUACION`, que ya resume su relacion como ratio.
- `NUM_ACTIVIDADES_POLITECNICAS` vs `NUM_ACTIVIDADES_T1/T2/T3`: total vs. desagregacion
  por termino, aporta granularidad temporal.
- `NUM_PUBLICACIONES` vs `NUM_PUBLICACIONES_INDEXADAS`: total vs. subconjunto de calidad.

Este mismo analisis ya fue documentado en `data/features/feature_correlations.csv`
(notebook 03); la tabla de esta seccion (`correlaciones_features.csv`) lo confirma
sobre el bloque ya imputado/transformado que efectivamente entra a `X_modelado`.

In [ ]:
dup_practico = correlaciones_features[correlaciones_features["POSIBLE_DUPLICADO_PRACTICO"]]
print("Pares con corr >= 0.99 (candidatos a duplicado practico):", len(dup_practico))
dup_practico

## 12. Variables de escala temporal

Las fechas crudas ya fueron excluidas en `03_construccion_features.ipynb`
(`FECHA_PRIMER_INGRESO`, `FECHA_ULTIMO_PERIODO_FIN`, `FECHANACIMIENTO`, ver
`features_excluded.csv`). El dataset de entrada ya representa el tiempo unicamente a
traves de variables derivadas (`ANTIGUEDAD_EFECTIVA_ANIOS`, `ANTIGUEDAD_CALENDARIO_ANIOS`,
`ANIOS_EXPERIENCIA_DOCENTE`, `ANIOS_EXPERIENCIA_ADMINISTRATIVO`). No se reconstruye
ninguna feature temporal en este notebook.

In [ ]:
fecha_cols = [c for c in df.columns if "FECHA" in c.upper()]
print("Columnas de fecha cruda en el dataset de entrada:", fecha_cols)
assert fecha_cols == []

## 8. Escalamiento

Se compara `StandardScaler` frente a `RobustScaler` **sobre los datos reales ya
imputados y transformados** (numericas + ordinal), en vez de asumir una opcion por
defecto.

In [ ]:
SCALED_NUMERIC_COLS = NUMERIC_COLS + ["NIVEL_ACADEMICO_MAXIMO_ORD"]
assert df_prep[SCALED_NUMERIC_COLS].isna().sum().sum() == 0
assert np.isfinite(df_prep[SCALED_NUMERIC_COLS].to_numpy()).all()

q1 = df_prep[SCALED_NUMERIC_COLS].quantile(.25)
q3 = df_prep[SCALED_NUMERIC_COLS].quantile(.75)
cols_iqr_cero = (q3 - q1)[(q3 - q1) == 0].index.tolist()

std_check = StandardScaler().fit_transform(df_prep[SCALED_NUMERIC_COLS])
rob_check = RobustScaler().fit_transform(df_prep[SCALED_NUMERIC_COLS])

comparacion_escalado = pd.DataFrame({
    "Metrica": ["Columnas con IQR=0 (tras imputacion/log1p)", "Valor absoluto maximo (StandardScaler)",
                "Valor absoluto maximo (RobustScaler)", "% valores con |z| > 3 (StandardScaler)",
                "% valores con |z| > 3 (RobustScaler)"],
    "Valor": [f"{len(cols_iqr_cero)} / {len(SCALED_NUMERIC_COLS)}", round(np.abs(std_check).max(), 2),
              round(np.abs(rob_check).max(), 2), round((np.abs(std_check) > 3).mean() * 100, 2),
              round((np.abs(rob_check) > 3).mean() * 100, 2)],
})
comparacion_escalado

**Decision: `StandardScaler`.**

Tras imputar y aplicar `log1p`, ~27% de las columnas numericas (variables muy
zero-infladas, y `PROMEDIO_HETEROEVALUACION` cuya mediana coincide con Q1 y Q3 por la
imputacion) quedan con **IQR = 0**. `RobustScaler` de scikit-learn, al dividir por un
IQR de 0, deja esas columnas sin normalizar (`scale_=1`), lo que produce valores
escalados muy por fuera del rango del resto de la matriz (hasta ~45 vs ~1-9). Dado que
`log1p` ya redujo la mayor parte de la asimetria y los outliers extremos (seccion 5),
`StandardScaler` ofrece una normalizacion consistente en todas las columnas sin ese
efecto colateral.

In [ ]:
scaler_elegido = StandardScaler()
print("Scaler seleccionado:", scaler_elegido.__class__.__name__)

## 13. Categoricas de alta cardinalidad

Ver seccion 3: `CARGO_ACTUAL`, `CARGO_MAS_FRECUENTE` y `UNIDAD_ACTUAL_NOMBRE` superan
el umbral de 20 categorias y quedan fuera de `X_modelado`, documentadas en
`modelado_variables_excluidas.csv` para revision/tratamiento posterior.

## 14. Matriz final

`ColumnTransformer` aplica, sobre `df_prep` (copia imputada y transformada, el
dataset original no se modifica):

- `SCALED_NUMERIC_COLS` (numericas + ordinal codificada) -> `StandardScaler`
- `NOMINAL_COLS` -> `OneHotEncoder` (categoria `DESCONOCIDO` ya incluida como una
  categoria mas)
- `BINARY_COLS` -> passthrough (ya en 0/1, no se re-escalan ni se one-hot-codifican)

`IDPERSONA` no entra al transformador; se conserva por separado en `personas_ids`.

In [ ]:
preprocesador = ColumnTransformer(transformers=[
    ("num", scaler_elegido, SCALED_NUMERIC_COLS),
    ("nom", OneHotEncoder(handle_unknown="ignore", sparse_output=False), NOMINAL_COLS),
    ("bin", "passthrough", BINARY_COLS),
], remainder="drop")

X = preprocesador.fit_transform(df_prep)
nombres_columnas_X = preprocesador.get_feature_names_out()

X_modelado = pd.DataFrame(X, columns=nombres_columnas_X)
personas_ids = df[["IDPERSONA"]].reset_index(drop=True)
personas_ids["INDICE_X_MODELADO"] = personas_ids.index

print("X_modelado:", X_modelado.shape)
X_modelado.head()

## 15. Guardar resultados

In [ ]:
def fuente_original(nombre_col):
    prefix, _, resto = nombre_col.partition("__")
    if prefix == "num":
        return "NIVEL_ACADEMICO_MAXIMO" if resto == "NIVEL_ACADEMICO_MAXIMO_ORD" else resto
    if prefix == "nom":
        return next(c for c in NOMINAL_COLS if resto.startswith(c + "_"))
    return resto  # bin

feature_names_modelado = pd.DataFrame([{
    "FEATURE": nombre,
    "TIPO": ("ordinal" if nombre == "num__NIVEL_ACADEMICO_MAXIMO_ORD"
             else "numerica" if nombre.startswith("num__")
             else "categorica_nominal" if nombre.startswith("nom__")
             else "binaria"),
    "TRANSFORMACION": ("log1p+standardscaler" if fuente_original(nombre) in LOG1P_COLS
                        else "standardscaler" if nombre.startswith("num__")
                        else "one_hot_encoding" if nombre.startswith("nom__")
                        else "ninguna"),
    "ESCALAMIENTO": "StandardScaler" if nombre.startswith("num__") else "ninguno (0/1)",
    "FUENTE": fuente_original(nombre),
} for nombre in nombres_columnas_X])

feature_names_modelado.to_csv(MODELING_DIR / "feature_names_modelado.csv", index=False)
feature_names_modelado

In [ ]:
def fila_resumen(c):
    if c in HIGH_CARD_COLS:
        return False, "DESCONOCIDO (documentada, no aplicada a la matriz)", "ninguna", \
            "ninguno (excluida por alta cardinalidad)", "ninguno"
    if c in NUMERIC_COLS:
        imput = "0" if c in ZERO_IMPUTE else ("mediana" if c in MEDIAN_IMPUTE else "sin nulos")
        transf = "log1p" if c in LOG1P_COLS else "ninguna"
        return True, imput, transf, "ninguno (numerica)", "StandardScaler"
    if c in ORDINAL_COLS:
        return True, "sin nulos", "ninguna", "ordinal explicito", "StandardScaler"
    if c in NOMINAL_COLS:
        return True, "DESCONOCIDO", "ninguna", "one_hot", "ninguno (dummy 0/1)"
    if c in BINARY_COLS:
        return True, "moda", "ninguna", "binaria 0/1", "ninguno (0/1)"
    raise ValueError(c)

preprocessing_summary = pd.DataFrame([{
    "FEATURE": c, "TIPO_ORIGINAL": str(df[c].dtype), "NULOS": round(df[c].isna().mean() * 100, 2),
    "IMPUTACION": fila_resumen(c)[1], "TRANSFORMACION": fila_resumen(c)[2],
    "ENCODING": fila_resumen(c)[3], "ESCALAMIENTO": fila_resumen(c)[4], "INCLUIDA": fila_resumen(c)[0],
} for c in df.columns if c != ID_COL])

preprocessing_summary.to_csv(MODELING_DIR / "preprocessing_summary.csv", index=False)
preprocessing_summary

In [ ]:
X_modelado.to_csv(MODELING_DIR / "X_modelado.csv", index=False)
personas_ids.to_csv(MODELING_DIR / "personas_modelado.csv", index=False)
joblib.dump(preprocesador, MODELING_DIR / "preprocessor.joblib")

print("Archivos guardados en", MODELING_DIR)
for nombre in ["X_modelado.csv", "personas_modelado.csv", "feature_names_modelado.csv",
               "preprocessing_summary.csv", "preprocessor.joblib"]:
    print(" -", nombre)

## 18. Validaciones finales

In [ ]:
assert df["IDPERSONA"].is_unique
assert X_modelado.shape[0] == df.shape[0] == personas_ids.shape[0]
assert not X_modelado.isna().any().any()
assert np.isfinite(X_modelado.to_numpy()).all()
assert all(pd.api.types.is_numeric_dtype(t) for t in X_modelado.dtypes)
assert (personas_ids["IDPERSONA"].values == df["IDPERSONA"].values).all()

print("Todas las validaciones pasaron.")
print("Personas:", X_modelado.shape[0])
print("Features originales (sin IDPERSONA):", df.shape[1] - 1)
print("Features excluidas (alta cardinalidad):", len(HIGH_CARD_COLS))
print("Features numericas de entrada:", len(NUMERIC_COLS))
print("Features categoricas de entrada (nominal + ordinal):", len(NOMINAL_COLS) + len(ORDINAL_COLS))
print("Features binarias de entrada:", len(BINARY_COLS))
print("Features numericas transformadas con log1p:", len(LOG1P_COLS))
print("Features categoricas codificadas (One-Hot, nominales):", len(NOMINAL_COLS))
print("Features finales en X_modelado:", X_modelado.shape[1])

## 19. Informe de preparacion

In [ ]:
informe = pd.DataFrame({
    "Metrica": [
        "Personas", "Features originales", "Features excluidas", "Features numericas",
        "Features categoricas (nominal+ordinal)", "Features binarias", "Features transformadas (log1p)",
        "Features despues de encoding", "Features finales", "Nulos finales", "Infinitos finales",
    ],
    "Valor": [
        X_modelado.shape[0], df.shape[1] - 1, len(HIGH_CARD_COLS), len(NUMERIC_COLS),
        len(NOMINAL_COLS) + len(ORDINAL_COLS), len(BINARY_COLS), len(LOG1P_COLS),
        X_modelado.shape[1], X_modelado.shape[1], int(X_modelado.isna().sum().sum()),
        int((~np.isfinite(X_modelado.to_numpy())).sum()),
    ],
})
informe